# User Segmentation with RFM Analysis
### A Transferable Framework for Gaming, SaaS, E-Commerce & Marketing Analytics

---

## Business Context

Understanding **who your users are** — not just in aggregate, but in behaviorally distinct groups — is one of the highest-leverage analyses an analyst can deliver. Whether you're at a gaming studio, a subscription SaaS company, a retailer, or a fintech platform, the core question is the same:

> *Which users are most valuable, which are at risk of leaving, and which are already gone — and what do we do about each group?*

**RFM segmentation** answers this question by grouping users along three dimensions:

| Dimension | Definition | Gaming Example | General Example |
|---|---|---|---|
| **Recency (R)** | How recently did the user engage/transact? | Days since last match played | Days since last purchase |
| **Frequency (F)** | How often do they engage? | Number of sessions in past 90 days | Number of orders in past 90 days |
| **Monetary (M)** | How much value have they generated? | Total in-app spend (USD) | Total revenue (USD) |

### What This Notebook Demonstrates
- Data cleaning and feature engineering
- RFM metric calculation and quintile scoring
- K-Means clustering with elbow method for optimal `k`
- Segment profiling and visualization
- Actionable business recommendations per segment

**Dataset:** Synthetic player/user transaction data (mirrors structure of UCI Online Retail II, Kaggle mobile game datasets, or any CRM export)

---

## 0. Setup & Imports

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
from datetime import datetime, timedelta
import warnings
warnings.filterwarnings('ignore')

# Plot styling
sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.dpi'] = 120
plt.rcParams['font.family'] = 'sans-serif'

SNAPSHOT_DATE = datetime(2024, 12, 31)  # The 'today' reference for recency calculation

print('Libraries loaded successfully.')

---
## 1. Data Generation

We generate a realistic synthetic dataset representing user transaction/session activity over 12 months. The data mimics what you'd pull from a CRM, game telemetry pipeline, or data warehouse.

**Schema:**
- `user_id` — unique user/player identifier
- `transaction_date` — date of purchase or meaningful session event
- `amount` — revenue generated (USD); 0 for free-to-play sessions, positive for purchases

> **Note:** In a real project, replace this cell with your data ingestion step (SQL query, Kaggle CSV load, API pull, etc.)

In [ ]:
np.random.seed(42)

N_USERS = 5000
START_DATE = datetime(2024, 1, 1)
END_DATE = datetime(2024, 12, 31)
DATE_RANGE = (END_DATE - START_DATE).days

# Define user archetypes with different behavioral profiles
archetypes = {
    'champion':      {'n': 400,  'freq_mu': 45, 'freq_sd': 8,  'spend_mu': 85,  'spend_sd': 30,  'recency_max': 15},
    'loyal':         {'n': 700,  'freq_mu': 25, 'freq_sd': 6,  'spend_mu': 40,  'spend_sd': 15,  'recency_max': 30},
    'potential':     {'n': 900,  'freq_mu': 12, 'freq_sd': 4,  'spend_mu': 20,  'spend_sd': 10,  'recency_max': 45},
    'at_risk':       {'n': 800,  'freq_mu': 18, 'freq_sd': 5,  'spend_mu': 35,  'spend_sd': 12,  'recency_max': 120},
    'hibernating':   {'n': 700,  'freq_mu': 8,  'freq_sd': 3,  'spend_mu': 15,  'spend_sd': 8,   'recency_max': 200},
    'lost':          {'n': 500,  'freq_mu': 3,  'freq_sd': 2,  'spend_mu': 5,   'spend_sd': 4,   'recency_max': 340},
    'new_user':      {'n': 1000, 'freq_mu': 4,  'freq_sd': 2,  'spend_mu': 10,  'spend_sd': 8,   'recency_max': 20},
}

records = []
user_id = 1

for archetype, params in archetypes.items():
    for _ in range(params['n']):
        n_transactions = max(1, int(np.random.normal(params['freq_mu'], params['freq_sd'])))
        recency_offset = np.random.randint(1, params['recency_max'])
        last_date = END_DATE - timedelta(days=recency_offset)

        for t in range(n_transactions):
            days_offset = np.random.randint(0, min(DATE_RANGE, params['recency_max'] + 90))
            txn_date = last_date - timedelta(days=days_offset)
            txn_date = max(txn_date, START_DATE)  # clamp to start of year
            amount = max(0, np.random.normal(params['spend_mu'], params['spend_sd']))
            records.append({'user_id': f'U{user_id:05d}', 'transaction_date': txn_date, 'amount': round(amount, 2)})

        user_id += 1

df_raw = pd.DataFrame(records)
df_raw['transaction_date'] = pd.to_datetime(df_raw['transaction_date'])

print(f'Dataset shape: {df_raw.shape}')
print(f'Unique users: {df_raw["user_id"].nunique():,}')
print(f'Date range: {df_raw["transaction_date"].min().date()} → {df_raw["transaction_date"].max().date()}')
print(f'Total revenue: ${df_raw["amount"].sum():,.2f}')
df_raw.head(10)

---
## 2. Exploratory Data Analysis (EDA)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4))

# Transaction volume over time
df_raw.set_index('transaction_date').resample('W')['user_id'].count().plot(
    ax=axes[0], color='steelblue', linewidth=1.5)
axes[0].set_title('Weekly Transaction Volume', fontweight='bold')
axes[0].set_xlabel('Date')
axes[0].set_ylabel('Transactions')

# Transaction amount distribution (log scale)
df_raw[df_raw['amount'] > 0]['amount'].hist(bins=50, ax=axes[1], color='coral', edgecolor='white')
axes[1].set_title('Transaction Amount Distribution', fontweight='bold')
axes[1].set_xlabel('Amount (USD)')
axes[1].set_ylabel('Count')

# Transactions per user
df_raw.groupby('user_id')['transaction_date'].count().hist(bins=40, ax=axes[2], color='mediumseagreen', edgecolor='white')
axes[2].set_title('Transactions per User', fontweight='bold')
axes[2].set_xlabel('Number of Transactions')
axes[2].set_ylabel('Users')

plt.tight_layout()
plt.suptitle('EDA Overview', y=1.02, fontsize=13, fontweight='bold')
plt.show()

---
## 3. RFM Feature Engineering

We aggregate the raw transaction log to one row per user with three computed metrics.

In [ ]:
rfm = df_raw.groupby('user_id').agg(
    last_transaction=('transaction_date', 'max'),
    frequency=('transaction_date', 'count'),
    monetary=('amount', 'sum')
).reset_index()

rfm['recency'] = (SNAPSHOT_DATE - rfm['last_transaction']).dt.days
rfm.drop(columns=['last_transaction'], inplace=True)

print('RFM table shape:', rfm.shape)
print('\nDescriptive statistics:')
rfm[['recency', 'frequency', 'monetary']].describe().round(2)

### 3.1 RFM Quintile Scoring

Each user receives a score of **1–5** for each RFM dimension based on quintile rank.

- For **Recency**: lower days = better → score 5 is most recent
- For **Frequency** and **Monetary**: higher = better → score 5 is highest

These scores are also useful standalone as features in downstream models (churn prediction, LTV regression, etc.).

In [ ]:
# Recency: lower days = better, so reverse scoring
rfm['R_score'] = pd.qcut(rfm['recency'], q=5, labels=[5, 4, 3, 2, 1]).astype(int)

# Frequency and Monetary: higher = better
rfm['F_score'] = pd.qcut(rfm['frequency'].rank(method='first'), q=5, labels=[1, 2, 3, 4, 5]).astype(int)
rfm['M_score'] = pd.qcut(rfm['monetary'].rank(method='first'), q=5, labels=[1, 2, 3, 4, 5]).astype(int)

# Composite RFM score (simple average; can weight differently per use case)
rfm['RFM_score'] = (rfm['R_score'] + rfm['F_score'] + rfm['M_score']) / 3

print('RFM scores added. Score distribution:')
rfm[['R_score', 'F_score', 'M_score', 'RFM_score']].describe().round(2)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
score_cols = ['R_score', 'F_score', 'M_score']
colors = ['#4C72B0', '#DD8452', '#55A868']
labels = ['Recency Score', 'Frequency Score', 'Monetary Score']

for ax, col, color, label in zip(axes, score_cols, colors, labels):
    rfm[col].value_counts().sort_index().plot(kind='bar', ax=ax, color=color, edgecolor='white')
    ax.set_title(label, fontweight='bold')
    ax.set_xlabel('Score (1=Worst, 5=Best)')
    ax.set_ylabel('Users')
    ax.tick_params(axis='x', rotation=0)

plt.suptitle('RFM Score Distributions', fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

---
## 4. K-Means Clustering

RFM quintile scores give us a numeric representation of each user. We apply K-Means to find **natural groupings** in this 3D space.

### 4.1 Find Optimal K — Elbow Method + Silhouette Score

In [ ]:
features = rfm[['R_score', 'F_score', 'M_score']].values
scaler = StandardScaler()
features_scaled = scaler.fit_transform(features)

inertias = []
silhouette_scores = []
k_range = range(2, 11)

for k in k_range:
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    km.fit(features_scaled)
    inertias.append(km.inertia_)
    silhouette_scores.append(silhouette_score(features_scaled, km.labels_))

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

axes[0].plot(list(k_range), inertias, 'o-', color='steelblue', linewidth=2)
axes[0].set_title('Elbow Method — Inertia vs. K', fontweight='bold')
axes[0].set_xlabel('Number of Clusters (k)')
axes[0].set_ylabel('Inertia')
axes[0].axvline(x=5, color='red', linestyle='--', alpha=0.6, label='Selected k=5')
axes[0].legend()

axes[1].plot(list(k_range), silhouette_scores, 'o-', color='coral', linewidth=2)
axes[1].set_title('Silhouette Score vs. K', fontweight='bold')
axes[1].set_xlabel('Number of Clusters (k)')
axes[1].set_ylabel('Silhouette Score')
axes[1].axvline(x=5, color='red', linestyle='--', alpha=0.6, label='Selected k=5')
axes[1].legend()

plt.tight_layout()
plt.show()

print(f'Silhouette scores: {dict(zip(k_range, [round(s,3) for s in silhouette_scores]))}')

### 4.2 Fit Final Model (k=5)

In [ ]:
K_OPTIMAL = 5
km_final = KMeans(n_clusters=K_OPTIMAL, random_state=42, n_init=10)
rfm['cluster'] = km_final.fit_predict(features_scaled)

# Profile each cluster by mean RFM values
cluster_profile = rfm.groupby('cluster').agg(
    n_users=('user_id', 'count'),
    avg_recency=('recency', 'mean'),
    avg_frequency=('frequency', 'mean'),
    avg_monetary=('monetary', 'mean'),
    avg_rfm_score=('RFM_score', 'mean')
).round(1)

cluster_profile['pct_users'] = (cluster_profile['n_users'] / cluster_profile['n_users'].sum() * 100).round(1)
cluster_profile.sort_values('avg_rfm_score', ascending=False)

### 4.3 Assign Segment Labels

We map each cluster to a human-readable label based on its behavioral profile. The labels are **intentionally domain-agnostic** — they apply equally to gaming, SaaS, e-commerce, or financial services.

In [ ]:
# Sort clusters by composite RFM score to assign labels in order
cluster_rank = rfm.groupby('cluster')['RFM_score'].mean().sort_values(ascending=False)
rank_to_label = {
    0: 'Champions',
    1: 'Loyal Users',
    2: 'At Risk',
    3: 'Dormant',
    4: 'Lost'
}
cluster_id_to_label = {cluster_id: rank_to_label[rank] for rank, cluster_id in enumerate(cluster_rank.index)}
rfm['segment'] = rfm['cluster'].map(cluster_id_to_label)

print('Segment distribution:')
print(rfm['segment'].value_counts().to_string())

---
## 5. Segment Profiling & Visualization

In [ ]:
# Segment summary table
segment_summary = rfm.groupby('segment').agg(
    Users=('user_id', 'count'),
    Avg_Recency_Days=('recency', 'mean'),
    Avg_Frequency=('frequency', 'mean'),
    Avg_Monetary=('monetary', 'mean'),
    Total_Revenue=('monetary', 'sum')
).round(1)

segment_summary['% of Users'] = (segment_summary['Users'] / segment_summary['Users'].sum() * 100).round(1)
segment_summary['% of Revenue'] = (segment_summary['Total_Revenue'] / segment_summary['Total_Revenue'].sum() * 100).round(1)
segment_summary = segment_summary.loc[['Champions', 'Loyal Users', 'At Risk', 'Dormant', 'Lost']]

print('=== Segment Summary ===')
segment_summary

In [ ]:
SEGMENT_COLORS = {
    'Champions':  '#2ecc71',
    'Loyal Users':'#3498db',
    'At Risk':    '#f39c12',
    'Dormant':    '#e67e22',
    'Lost':       '#e74c3c'
}

fig, axes = plt.subplots(2, 2, figsize=(15, 12))

# ── 1. Scatter: Frequency vs Recency, colored by segment ──
ax = axes[0, 0]
for seg, grp in rfm.groupby('segment'):
    ax.scatter(grp['recency'], grp['frequency'], alpha=0.35, s=12,
               color=SEGMENT_COLORS[seg], label=seg)
ax.set_xlabel('Recency (days since last activity)')
ax.set_ylabel('Frequency (# transactions)')
ax.set_title('Recency vs. Frequency by Segment', fontweight='bold')
ax.legend(markerscale=2, fontsize=9)

# ── 2. Bar: Revenue share by segment ──
ax = axes[0, 1]
rev_data = segment_summary['% of Revenue'].loc[['Champions', 'Loyal Users', 'At Risk', 'Dormant', 'Lost']]
bars = ax.barh(rev_data.index, rev_data.values,
               color=[SEGMENT_COLORS[s] for s in rev_data.index], edgecolor='white')
ax.set_xlabel('% of Total Revenue')
ax.set_title('Revenue Share by Segment', fontweight='bold')
for bar, val in zip(bars, rev_data.values):
    ax.text(val + 0.3, bar.get_y() + bar.get_height()/2, f'{val}%', va='center', fontsize=9)
ax.set_xlim(0, rev_data.max() + 8)

# ── 3. Heatmap: Mean RFM scores per segment ──
ax = axes[1, 0]
heatmap_data = rfm.groupby('segment')[['R_score', 'F_score', 'M_score']].mean()
heatmap_data = heatmap_data.loc[['Champions', 'Loyal Users', 'At Risk', 'Dormant', 'Lost']]
heatmap_data.columns = ['Recency\nScore', 'Frequency\nScore', 'Monetary\nScore']
sns.heatmap(heatmap_data, annot=True, fmt='.1f', cmap='RdYlGn',
            vmin=1, vmax=5, linewidths=0.5, ax=ax, cbar_kws={'label': 'Score (1=Low, 5=High)'})
ax.set_title('Mean RFM Scores by Segment', fontweight='bold')
ax.set_ylabel('')

# ── 4. Donut: User count by segment ──
ax = axes[1, 1]
user_data = segment_summary['Users'].loc[['Champions', 'Loyal Users', 'At Risk', 'Dormant', 'Lost']]
wedges, texts, autotexts = ax.pie(
    user_data.values,
    labels=user_data.index,
    colors=[SEGMENT_COLORS[s] for s in user_data.index],
    autopct='%1.1f%%',
    startangle=140,
    wedgeprops=dict(width=0.5)
)
for t in autotexts:
    t.set_fontsize(9)
ax.set_title('User Distribution by Segment', fontweight='bold')

plt.suptitle('RFM Segmentation — Full Dashboard', fontsize=15, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

---
## 6. Business Recommendations

RFM segmentation only earns its keep when it drives action. Below are recommended strategies per segment, framed for two contexts to demonstrate transferability.

---

### 🟢 Champions — High R, High F, High M
*These are your most valuable, most active, most recent users.*

| Context | Recommended Action |
|---|---|
| **Gaming (Riot)** | Early access to new content, exclusive cosmetics, beta tester programs. Avoid over-monetizing — protect LTV. |
| **SaaS / E-Commerce** | Loyalty rewards, referral programs, upsell to premium tier, NPS survey candidates. |

**KPI to track:** Retention rate, referral rate, LTV growth

---

### 🔵 Loyal Users — Moderate-High across all three
*Consistent, engaged users who haven't yet hit Champion tier.*

| Context | Recommended Action |
|---|---|
| **Gaming (Riot)** | Streak rewards, Battle Pass nudges, social features to deepen engagement loops. |
| **SaaS / E-Commerce** | Feature education (are they using the full product?), loyalty tier upgrades, targeted upsell emails. |

**KPI to track:** Feature adoption rate, session depth, conversion to higher spend tier

---

### 🟡 At Risk — Previously high engagement, declining recency
*Used to be active. Something changed. Act fast.*

| Context | Recommended Action |
|---|---|
| **Gaming (Riot)** | Re-engagement push notifications ('Your friends are playing'), patch highlight emails, free XP boost. |
| **SaaS / E-Commerce** | Win-back email sequence, personalized 'We miss you' offer, churn risk flag to CS team. |

**KPI to track:** Reactivation rate, campaign CTR, 30-day retention post-campaign

---

### 🟠 Dormant — Low recency, low frequency, low spend
*Present but not engaged. Low cost to re-engage, lower probability of success.*

| Context | Recommended Action |
|---|---|
| **Gaming (Riot)** | Major content update announcements, seasonal event emails, low-friction re-entry point. |
| **SaaS / E-Commerce** | Broad newsletter, promotional discount, segment for suppression if no response in 60 days. |

**KPI to track:** Open rate, reactivation rate, cost per reactivated user

---

### 🔴 Lost — Very low recency and activity, minimal spend
*Effectively churned. ROI of re-engagement is low.*

| Context | Recommended Action |
|---|---|
| **Gaming (Riot)** | One final 'comeback' campaign tied to a major release. If no response, suppress to protect sender reputation. |
| **SaaS / E-Commerce** | Unsubscribe suppression, GDPR/data hygiene candidate, analyze exit survey data to identify root cause. |

**KPI to track:** Response rate to final campaign, cost of suppression vs. continued mailing

---

## 7. Analytical Extensions

This notebook is a foundation. Here are natural next steps that extend the portfolio value:

1. **Churn Prediction Model** — Use RFM scores + segment as features in an XGBoost classifier to predict 30-day churn probability at the user level
2. **LTV Regression** — Predict future 90-day revenue per user using historical RFM + segment features
3. **A/B Test on Segment** — Design a hold-out experiment: send re-engagement campaign to 50% of 'At Risk' users and measure lift in reactivation
4. **Segment Migration Tracking** — Run RFM monthly, track how users move between segments over time (cohort flow diagram)
5. **Campaign ROI Calculator** — Attach estimated CAC/reactivation cost per segment to produce expected ROI per campaign send

---

## 8. Export Segmented Data

In [ ]:
output_cols = ['user_id', 'recency', 'frequency', 'monetary',
               'R_score', 'F_score', 'M_score', 'RFM_score', 'segment']

rfm[output_cols].sort_values('RFM_score', ascending=False).to_csv('rfm_segments_output.csv', index=False)

print('Exported rfm_segments_output.csv')
print(f'Total users segmented: {len(rfm):,}')
print('\nFinal segment counts:')
print(rfm['segment'].value_counts().to_string())